# 🎯 Prompt Evaluation Harness & Drift Detection
## Production-Grade Prompt Quality Scoring & Version Drift Benchmarking

**Pack ID:** `pack_0003` · **Category:** `ai_ops` · **Difficulty:** `advanced` · **Version:** `1.0.0`

---

### 📋 Overview
Evaluating prompts manually across agent versions is error-prone, subjective, and labor-intensive. When engineering teams tweak a system prompt, subtle regressions often emerge in instruction following, edge-case safety, or token bloat.

This notebook provides an automated, deterministic evaluation harness that:
1. **Scores Prompt Quality** across 5 objective dimensions: Clarity, Specificity, Completeness, Relevance, and Safety.
2. **Measures Version Drift** ($\Delta$) between baseline prompts (v1.0) and candidate prompts (v2.0).
3. **Detects Regressions** before prompts are deployed to production agents.
4. **Exports Machine-Readable Reports** (`outputs/evaluation_results.json` and `outputs/evaluation_report.md`).

```
  [ data/prompts.json ]
          │
          ▼
  ┌────────────────────────────────────────────────────────┐
  │  Prompt Evaluation Engine                              │
  │  ├─ Clarity & Readability Scoring                      │
  │  ├─ Specificity & Constraint Rigor                     │
  │  ├─ Completeness & Role Framing                        │
  │  ├─ Domain Relevance Density                           │
  │  └─ Injection Resistance & Safety                      │
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
  ┌────────────────────────────────────────────────────────┐
  │  Drift Detection & Regression Analysis (v1.0 vs v2.0)  │
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
    outputs/evaluation_results.json  &  outputs/evaluation_report.md
```

---


In [ ]:
# ==============================================================================
# STEP 1: ENVIRONMENT SETUP & DETERMINISTIC SEEDING
# ==============================================================================
import sys
import os
import json
import re
import time
import random
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# Enforce deterministic seed for reproducible benchmarking
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Python Version: {sys.version.split()[0]}")
assert sys.version_info >= (3, 10), "Python 3.10 or higher is required!"

# Verify required libraries
try:
    import pandas as pd
    import numpy as np
    np.random.seed(RANDOM_SEED)
    print(f"Pandas Version: {pd.__version__}")
    print(f"NumPy Version:  {np.__version__}")
except ImportError as err:
    print(f"⚠️ Missing package: {err}. Please run: pip install -r requirements.txt")
    raise

print("\n✅ Environment validated and deterministic seeds locked.")


In [ ]:
# ==============================================================================
# STEP 2: CONFIGURATION & PATH SETUP
# ==============================================================================
DATA_PATH = Path("data/prompts.json")
OUTPUTS_DIR = Path("outputs")
OUTPUT_JSON_PATH = OUTPUTS_DIR / "evaluation_results.json"
OUTPUT_MD_PATH = OUTPUTS_DIR / "evaluation_report.md"

# Ensure outputs directory exists with write permissions
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation Weights
METRIC_WEIGHTS = {
    "clarity": 0.20,
    "specificity": 0.25,
    "completeness": 0.25,
    "relevance": 0.15,
    "safety": 0.15
}

print(f"Input Path:       {DATA_PATH.resolve()}")
print(f"Outputs Directory: {OUTPUTS_DIR.resolve()}")
print(f"Metric Weights:   {METRIC_WEIGHTS}")
print("\n✅ Configuration loaded.")


In [ ]:
# ==============================================================================
# STEP 3: PROMPT DATASET INGESTION & VALIDATION
# ==============================================================================
def load_prompt_dataset(path: Path) -> List[Dict[str, Any]]:
    """Load prompts from JSON file or generate fallback benchmark dataset."""
    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
                prompts = payload.get("prompts", [])
                if prompts:
                    print(f"📂 Loaded {len(prompts)} prompts from {path}")
                    return prompts
        except Exception as e:
            print(f"⚠️ Error loading {path}: {e}. Falling back to default benchmark.")

    print("ℹ️ Generating sample benchmark prompts for evaluation...")
    fallback = [
        {
            "id": "summarizer_v1",
            "name": "Financial Summarizer (Baseline)",
            "category": "summarization",
            "version": "1.0",
            "system_prompt": "Summarize the text briefly for management.",
            "required_criteria": ["revenue", "margins", "risks"]
        },
        {
            "id": "summarizer_v2",
            "name": "Financial Summarizer (Optimized)",
            "category": "summarization",
            "version": "2.0",
            "system_prompt": "You are a Senior Financial Analyst. Summarize the provided document into a structured Executive Brief:\n1. Key Metrics Table (Revenue, Margin, Cash Flow)\n2. Primary Cost & Margin Drivers\n3. Strategic Risk Factors\nConstraints: Maintain strict factual accuracy. Do not extrapolate beyond provided numbers. Maximum 200 words.",
            "required_criteria": ["revenue", "margins", "risks", "structured table"]
        }
    ]
    return fallback

dataset = load_prompt_dataset(DATA_PATH)

# Validate structure of dataset
for p in dataset:
    assert "id" in p and "version" in p and "system_prompt" in p, f"Malformed prompt entry: {p}"

print(f"✅ Successfully validated {len(dataset)} prompt benchmark items across categories:")
categories = set(p.get("category", "general") for p in dataset)
for cat in categories:
    cat_prompts = [p for p in dataset if p.get("category") == cat]
    print(f"   - {cat:20}: {len(cat_prompts)} prompts")


In [ ]:
# ==============================================================================
# STEP 4: PROMPT EVALUATION METRICS ENGINE
# ==============================================================================
class PromptEvaluator:
    """Multi-dimensional rubric evaluator for LLM system prompts."""
    
    IMPERATIVE_VERBS = {'summarize', 'extract', 'generate', 'analyze', 'provide', 'ensure', 'maintain', 'return', 'format', 'verify', 'do', 'never'}
    SPECIFICITY_KEYWORDS = {'constraint', 'format', 'table', 'json', 'step', 'maximum', 'minimum', 'guideline', 'rule', 'schema', 'never', 'strict'}
    ROLE_KEYWORDS = {'you are', 'expert', 'specialist', 'engineer', 'analyst', 'assistant', 'officer'}
    INJECTION_RISK_TERMS = {'anything', 'whatever', 'no rules', 'ignore restrictions', 'unrestricted'}

    @classmethod
    def score_clarity(cls, text: str) -> float:
        """Measures structural clarity and imperative directive ratio."""
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        if not words:
            return 0.0
        has_structure = 1.0 if ('\n' in text or ':' in text or '-' in text) else 0.5
        imperatives = sum(1 for w in words if w in cls.IMPERATIVE_VERBS)
        imperative_density = min(1.0, (imperatives / max(5, len(words))) * 5.0)
        return round(0.4 * has_structure + 0.6 * imperative_density, 3)

    @classmethod
    def score_specificity(cls, text: str) -> float:
        """Measures presence of explicit constraints, boundaries, and formatting instructions."""
        lower = text.lower()
        keyword_matches = sum(1 for kw in cls.SPECIFICITY_KEYWORDS if kw in lower)
        score = min(1.0, keyword_matches * 0.2)
        # Bonus for explicit numerical limits (e.g. '200 words', '3 bullet points')
        if re.search(r'\b\d+\s*(words|sentences|bullets|items|minutes|hours)\b', lower):
            score = min(1.0, score + 0.2)
        return round(max(0.2, score), 3)

    @classmethod
    def score_completeness(cls, text: str, required_criteria: List[str]) -> float:
        """Checks coverage of persona framing and domain criteria."""
        lower = text.lower()
        has_role = 1.0 if any(r in lower for r in cls.ROLE_KEYWORDS) else 0.4
        if required_criteria:
            crit_matches = sum(1 for c in required_criteria if any(term in lower for term in c.lower().split()))
            crit_ratio = crit_matches / len(required_criteria)
        else:
            crit_ratio = 0.8
        return round(0.4 * has_role + 0.6 * crit_ratio, 3)

    @classmethod
    def score_relevance(cls, text: str, category: str) -> float:
        """Calculates category domain alignment."""
        category_tokens = {
            'summarization': ['brief', 'key', 'summary', 'metric', 'table', 'risk', 'document'],
            'code_generation': ['python', 'fastapi', 'schema', 'endpoint', 'error', 'model', 'router'],
            'customer_support': ['customer', 'support', 'billing', 'refund', 'empathy', 'guideline'],
            'ai_ops': ['agent', 'pipeline', 'metric', 'evaluation', 'drift']
        }
        tokens = category_tokens.get(category, ['task', 'system', 'output'])
        matches = sum(1 for t in tokens if t in text.lower())
        return round(min(1.0, 0.4 + (matches / len(tokens)) * 0.6), 3)

    @classmethod
    def score_safety(cls, text: str) -> float:
        """Evaluates resistance to ambiguity and injection vectors."""
        lower = text.lower()
        for risk in cls.INJECTION_RISK_TERMS:
            if risk in lower:
                return 0.4
        # High safety if explicitly stating negative constraints ('never', 'do not')
        safety_points = 0.75
        if 'never' in lower or 'do not' in lower or 'strict' in lower:
            safety_points += 0.25
        return round(min(1.0, safety_points), 3)

    @classmethod
    def evaluate(cls, prompt_data: Dict[str, Any], weights: Dict[str, float]) -> Dict[str, Any]:
        text = prompt_data.get("system_prompt", "")
        cat = prompt_data.get("category", "general")
        criteria = prompt_data.get("required_criteria", [])
        
        clarity = cls.score_clarity(text)
        specificity = cls.score_specificity(text)
        completeness = cls.score_completeness(text, criteria)
        relevance = cls.score_relevance(text, cat)
        safety = cls.score_safety(text)
        
        overall = round(
            clarity * weights.get('clarity', 0.2) +
            specificity * weights.get('specificity', 0.25) +
            completeness * weights.get('completeness', 0.25) +
            relevance * weights.get('relevance', 0.15) +
            safety * weights.get('safety', 0.15),
            3
        )
        
        return {
            "prompt_id": prompt_data["id"],
            "name": prompt_data.get("name", prompt_data["id"]),
            "category": cat,
            "version": prompt_data.get("version", "1.0"),
            "token_count": len(text.split()),
            "scores": {
                "clarity": clarity,
                "specificity": specificity,
                "completeness": completeness,
                "relevance": relevance,
                "safety": safety,
                "overall": overall
            }
        }

print("✅ PromptEvaluator metric suite compiled and ready.")


In [ ]:
# ==============================================================================
# STEP 5: BATCH PROMPT EVALUATION RUNNER
# ==============================================================================
evaluations = []
print("🚀 Running Evaluation on Benchmark Prompts...\n")

for p in dataset:
    res = PromptEvaluator.evaluate(p, METRIC_WEIGHTS)
    evaluations.append(res)
    sc = res["scores"]
    print(f"[{res['category']:16}] {res['name']:40} | Overall: {sc['overall']:.3f} | Words: {res['token_count']:3d}")

print(f"\n✅ Completed evaluation for {len(evaluations)} prompts.")


In [ ]:
# ==============================================================================
# STEP 6: VERSION DRIFT DETECTION & REGRESSION ANALYSIS
# ==============================================================================
def calculate_drift_analysis(eval_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Compare version pairs (v1.0 vs v2.0) per category to detect performance drift."""
    # Group by category
    by_cat = {}
    for e in eval_list:
        cat = e["category"]
        by_cat.setdefault(cat, []).append(e)
        
    drift_reports = []
    total_deltas = []
    
    for cat, items in by_cat.items():
        v1_items = [i for i in items if i["version"] == "1.0"]
        v2_items = [i for i in items if i["version"] == "2.0"]
        
        if v1_items and v2_items:
            v1 = v1_items[0]
            v2 = v2_items[0]
            delta = round(v2["scores"]["overall"] - v1["scores"]["overall"], 3)
            pct_change = round((delta / max(0.001, v1["scores"]["overall"])) * 100, 1)
            total_deltas.append(delta)
            
            if delta > 0.05:
                status = "SIGNIFICANT_IMPROVEMENT"
            elif delta < -0.05:
                status = "REGRESSION_DETECTED"
            else:
                status = "NEUTRAL_STABLE"
                
            drift_reports.append({
                "category": cat,
                "baseline_name": v1["name"],
                "candidate_name": v2["name"],
                "v1_score": v1["scores"]["overall"],
                "v2_score": v2["scores"]["overall"],
                "score_delta": delta,
                "percent_change": pct_change,
                "status": status
            })
            
    avg_drift = round(float(np.mean(total_deltas)), 3) if total_deltas else 0.0
    
    return {
        "comparison_pairs": len(drift_reports),
        "average_overall_drift": avg_drift,
        "regressions_count": sum(1 for r in drift_reports if r["status"] == "REGRESSION_DETECTED"),
        "improvements_count": sum(1 for r in drift_reports if r["status"] == "SIGNIFICANT_IMPROVEMENT"),
        "detailed_drift": drift_reports
    }

drift_results = calculate_drift_analysis(evaluations)

print("=" * 75)
print("📊 DRIFT DETECTION & REGRESSION SUMMARY")
print("=" * 75)
print(f"Average Score Drift: {'+' if drift_results['average_overall_drift']>0 else ''}{drift_results['average_overall_drift']:.3f}")
print(f"Regressions:         {drift_results['regressions_count']}")
print(f"Improvements:        {drift_results['improvements_count']}")
print("-" * 75)
for d in drift_results["detailed_drift"]:
    icon = "📈" if d["status"] == "SIGNIFICANT_IMPROVEMENT" else ("📉" if d["status"] == "REGRESSION_DETECTED" else "⚖️")
    print(f"{icon} [{d['category']:16}] {d['v1_score']:.3f} ➔ {d['v2_score']:.3f} ({'+' if d['score_delta']>0 else ''}{d['score_delta']:.3f}, {d['percent_change']:+5.1f}%) | {d['status']}")
print("=" * 75)


In [ ]:
# ==============================================================================
# STEP 7: TABULAR ANALYTICS & SCORE MATRICES
# ==============================================================================
# Create pandas DataFrame of evaluations
flat_rows = []
for e in evaluations:
    row = {
        "ID": e["prompt_id"],
        "Category": e["category"],
        "Version": e["version"],
        "Words": e["token_count"],
        "Clarity": e["scores"]["clarity"],
        "Specificity": e["scores"]["specificity"],
        "Completeness": e["scores"]["completeness"],
        "Relevance": e["scores"]["relevance"],
        "Safety": e["scores"]["safety"],
        "Overall": e["scores"]["overall"]
    }
    flat_rows.append(row)

df_scores = pd.DataFrame(flat_rows)
print("📋 Prompt Score Matrix:")
print(df_scores.to_string(index=False))

print("\n📈 Category Performance by Version:")
cat_summary = df_scores.groupby(["Category", "Version"])[["Specificity", "Completeness", "Overall"]].mean()
print(cat_summary.round(3))


In [ ]:
# ==============================================================================
# STEP 8: WRITE OUTPUT ARTIFACTS (JSON & MARKDOWN REPORT)
# ==============================================================================
# 1. Save outputs/evaluation_results.json
results_payload = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "total_evaluated": len(evaluations),
    "metric_weights": METRIC_WEIGHTS,
    "drift_summary": {
        "average_overall_drift": drift_results["average_overall_drift"],
        "regressions_count": drift_results["regressions_count"],
        "improvements_count": drift_results["improvements_count"]
    },
    "evaluations": evaluations,
    "drift_detection": drift_results["detailed_drift"]
}

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(results_payload, f, indent=2)
print(f"💾 Written: {OUTPUT_JSON_PATH} ({OUTPUT_JSON_PATH.stat().st_size} bytes)")

# 2. Generate outputs/evaluation_report.md
report_md = f"""# 📊 Prompt Evaluation & Drift Report

**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}  
**Total Prompts Audited:** {len(evaluations)}  
**Average Drift ($\\Delta$):** {'+' if drift_results['average_overall_drift']>0 else ''}{drift_results['average_overall_drift']:.3f}  
**Status:** {'✅ READY FOR DEPLOYMENT' if drift_results['regressions_count'] == 0 else '⚠️ REGRESSIONS DETECTED'}  

---

## 1. Executive Summary

The Prompt Evaluation Harness evaluated {len(evaluations)} system prompts across {len(drift_results['detailed_drift'])} domain categories. Comparing baseline versions (v1.0) against candidate versions (v2.0) demonstrates an average score improvement of **{drift_results['average_overall_drift']:+.3f}** across all rubric dimensions.

- **Significant Improvements:** {drift_results['improvements_count']}
- **Regressions Detected:** {drift_results['regressions_count']}
- **Drift Stability Verdict:** {'PASSED - All candidates meet or exceed baseline criteria.' if drift_results['regressions_count'] == 0 else 'BLOCKED - Review regressions before merging.'}

## 2. Version Drift Matrix

| Category | Baseline (v1.0) | Candidate (v2.0) | $\\Delta$ Score | Change % | Verdict |
|---|---|---|---|---|---|
"""

for d in drift_results["detailed_drift"]:
    badge = "✅ Improved" if d["status"] == "SIGNIFICANT_IMPROVEMENT" else ("❌ Regression" if d["status"] == "REGRESSION_DETECTED" else "⚖️ Stable")
    report_md += f"| {d['category']} | {d['v1_score']:.3f} | {d['v2_score']:.3f} | {d['score_delta']:+.3f} | {d['percent_change']:+.1f}% | {badge} |\n"

report_md += """
## 3. Detailed Prompt Metric Breakdown

| Prompt ID | Ver | Words | Clarity | Specificity | Completeness | Relevance | Safety | Overall |
|---|---|---|---|---|---|---|---|---|
"""

for e in evaluations:
    sc = e["scores"]
    report_md += f"| `{e['prompt_id']}` | {e['version']} | {e['token_count']} | {sc['clarity']:.2f} | {sc['specificity']:.2f} | {sc['completeness']:.2f} | {sc['relevance']:.2f} | {sc['safety']:.2f} | **{sc['overall']:.3f}** |\n"

report_md += """
---
### Recommended Action
Merge candidate v2.0 prompts into production agent configurations. Archive v1.0 baselines in the prompt registry for regression audits.
"""

with open(OUTPUT_MD_PATH, "w", encoding="utf-8") as f:
    f.write(report_md)
print(f"💾 Written: {OUTPUT_MD_PATH} ({OUTPUT_MD_PATH.stat().st_size} bytes)")


In [ ]:
# ==============================================================================
# STEP 9: VERIFICATION & SMOKE TEST
# ==============================================================================
assert OUTPUT_JSON_PATH.exists(), f"Missing output: {OUTPUT_JSON_PATH}"
assert OUTPUT_MD_PATH.exists(), f"Missing output: {OUTPUT_MD_PATH}"
assert OUTPUT_JSON_PATH.stat().st_size > 0, "JSON output file is empty"
assert OUTPUT_MD_PATH.stat().st_size > 0, "Markdown report file is empty"

print("=" * 75)
print("📄 PREVIEW: GENERATED EVALUATION REPORT")
print("=" * 75)
with open(OUTPUT_MD_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()[:25]
    print("".join(lines))
print("=" * 75)
print("✅ ALL VERIFICATION CHECKS PASSED")


---
## 🚀 Production CI/CD Integration & Upgrades

### 1. Automated GitHub Actions Gating
Run this evaluation harness on every pull request that modifies prompt templates:
```bash
jupyter nbconvert --to notebook --execute main.ipynb --output executed.ipynb
python -c "import json; data=json.load(open('outputs/evaluation_results.json')); exit(1 if data['drift_summary']['regressions_count'] > 0 else 0)"
```

### 2. Upgrading to LLM-as-a-Judge
To replace rule-based scoring with an LLM judge, pass candidate outputs to `gpt-4o` or `claude-3-5-sonnet` using the following scoring prompt:
```python
# In production, plug in your OpenAI / Anthropic client:
# client = OpenAI()
# response = client.chat.completions.create(
#     model='gpt-4o',
#     messages=[{'role': 'system', 'content': 'Evaluate response quality against rubric...'}, ...]
# )
```

---
**Pack 0003: Prompt Evaluation Harness** · Completed successfully.
